In [1]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e76dd8e6d97736f8fb9bdca6aee5a4643aa49d18f353cac44ad3321a024515b6
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
# dependencies
import torch
import torch.optim as optim
from transformers import BertForTokenClassification, BertTokenizerFast
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score as seqeval_f1
from seqeval.metrics import precision_score as seqeval_precision
from seqeval.metrics import recall_score as seqeval_recall
from tqdm.auto import tqdm  # modern, works in notebooks & scripts

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion")

print("Path to dataset files:", path)

base_path = path + "/"

100%|██████████| 960k/960k [00:00<00:00, 23.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/alaakhaled/conll003-englishversion/versions/1


In [4]:
# read the data files
def load_sentences(filepath):

    sentences = []
    tokens = []
    pos_tags = []
    chunk_tags = []
    ner_tags = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            # sentence boundary
            if (line.startswith('-DOCSTART-') or line.strip() == ''):
                if len(tokens) > 0:
                    sentences.append({
                        'tokens': tokens,
                        'pos_tags': pos_tags,
                        'chunk_tags': chunk_tags,
                        'ner_tags': ner_tags
                    })
                    tokens = []
                    pos_tags = []
                    chunk_tags = []
                    ner_tags = []
            else:
                l = line.strip().split(' ')
                if len(l) >= 4:
                    tokens.append(l[0])
                    pos_tags.append(l[1])
                    chunk_tags.append(l[2])
                    ner_tags.append(l[3])
    # last sentence if file doesn't end with blank line
    if len(tokens) > 0:
        sentences.append({
            'tokens': tokens,
            'pos_tags': pos_tags,
            'chunk_tags': chunk_tags,
            'ner_tags': ner_tags
        })
    return sentences

print('loading data')
train_sentences = load_sentences(base_path + 'train.txt')
test_sentences = load_sentences(base_path + 'test.txt')
valid_sentences = load_sentences(base_path + 'valid.txt')

loading data


In [6]:
train_sentences[0:2]

[{'tokens': ['EU',
   'rejects',
   'German',
   'call',
   'to',
   'boycott',
   'British',
   'lamb',
   '.'],
  'pos_tags': ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.'],
  'chunk_tags': ['B-NP',
   'B-VP',
   'B-NP',
   'I-NP',
   'B-VP',
   'I-VP',
   'B-NP',
   'I-NP',
   'O'],
  'ner_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']},
 {'tokens': ['Peter', 'Blackburn'],
  'pos_tags': ['NNP', 'NNP'],
  'chunk_tags': ['B-NP', 'I-NP'],
  'ner_tags': ['B-PER', 'I-PER']}]

In [5]:
# build tag set and label mappings
all_tags = sorted({tag for s in train_sentences for tag in s['ner_tags']})
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}
num_labels = len(all_tags)
print('Tagset size:', num_labels)
print('Tags:', all_tags)
print(f"Tag IDs: {id2label}")
print(f"Label2ID: {label2id}")

Tagset size: 9
Tags: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
Tag IDs: {0: 'B-LOC', 1: 'B-MISC', 2: 'B-ORG', 3: 'B-PER', 4: 'I-LOC', 5: 'I-MISC', 6: 'I-ORG', 7: 'I-PER', 8: 'O'}
Label2ID: {'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}


In [6]:
# load BERT tokenizer
bert_version = 'bert-base-uncased'
tokenizer = BertTokenizerFast.from_pretrained(bert_version)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
%%capture
# map tokens and tags to token ids and label ids
def align_label(tokens, labels, sentence):
    """
    It maps:
      - O means the word doesn’t correspond to any entity.
      - B-PER/I-PER means the word corresponds to the beginning of/is inside a person entity.
      - B-ORG/I-ORG means the word corresponds to the beginning of/is inside an organization entity.
      - B-LOC/I-LOC means the word corresponds to the beginning of/is inside a location entity.
      - B-MISC/I-MISC means the word corresponds to the beginning of/is inside a miscellaneous entity.
    to each token in the sentence.

    When it runs out of tokens, it adds -100 to the label. Each word index that does not correspond to a word
    in the sentence, gets a -100. Each unknown token gets a -100.

    INPUTS:

    OUTPUT:
      - labels: 512 length of labels for each token in the sentence.
      - tokens: tokenizer output (BatchEncoding) with word_ids()
      - labels: list of original word-level labels for a single sentence

    INFO:
    512: is the model output size.
    """
    word_ids = tokens.word_ids()
    # print(f"Word IDS: {word_ids}")
    previous_word_idx = None
    label_ids = []
    # print(tokens)

    for word_idx in word_ids:
        # print(f"\n===\n\tword idx: {word_idx}")

        # Debug
        # if word_idx is not None:
        #     print(f"Sentence token: {sentence["tokens"][word_idx]}")
        #     print(f"Sentence label: {sentence["ner_tags"][word_idx]}")

        if word_idx is None:
            # special tokens
            label_ids.append(-100)
            # print(f"Special token: {label_ids}")

        elif word_idx != previous_word_idx:
            # first subword of a word
            # Get the corresponding token NER tag from the labels or -100
            label_ids.append(label2id.get(labels[word_idx], -100))
            # print(f"First subword: {label_ids}")

        else:
            # subsequent subword of the same word (ignore for loss)
            label_ids.append(-100)
            # print(f"Subsequent subword: {label_ids}")

        previous_word_idx = word_idx

    # print(f"Word IDX: {word_idx}, labels: {label_ids}")

    return label_ids

def encode(sentence):
    encodings = tokenizer(
        sentence['tokens'],
        truncation=True,
        padding='max_length',
        is_split_into_words=True,
        return_tensors='pt'  # get tensors directly
    )
    # print(f"Encodings: {encodings}")
    labels = align_label(encodings, sentence['ner_tags'], sentence)
    # print(f"Final labels: {labels}")

    return {
        'input_ids': encodings['input_ids'].squeeze(0),        # [seq_len]
        'attention_mask': encodings['attention_mask'].squeeze(0),
        'labels': torch.tensor(labels, dtype=torch.long)
    }

print('encoding data')
# for sentence in train_sentences:
#     print(f"Sentence: {sentence}")
#     encode(sentence)
#     break
train_dataset = [encode(sentence) for sentence in train_sentences]
valid_dataset = [encode(sentence) for sentence in valid_sentences]
test_dataset = [encode(sentence) for sentence in test_sentences]

In [10]:
train_dataset[0]

{'input_ids': tensor([  101,  7327, 19164,  2446,  2655,  2000, 17757,  2329, 12559,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

In [11]:
torch.nn.functional.cross_entropy(
    # torch.tensor([15.], dtype=torch.long, requires_grad=True).float(),
    torch.randn(1, 1, requires_grad=True),
    torch.empty(1, dtype=torch.long).random_(1)
    # torch.tensor([1])
    )

tensor(0., grad_fn=<NllLossBackward0>)

In [12]:
input = torch.randn(1, 2, requires_grad=True)
# target = torch.empty(1, dtype=torch.long).random_(2)
target = torch.tensor([1], dtype=torch.long)

output = torch.nn.functional.cross_entropy(input, target)
print(f"Input: {input}")
print(f"target: {target}")
print(f"Output: {output}")

Input: tensor([[0.1930, 0.5774]], requires_grad=True)
target: tensor([1])
Output: 0.5192708969116211


In [8]:
# PyTorch Dataset wrapper for better compatibility with DataLoader
class InputDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = InputDataset(train_dataset)
valid_dataset = InputDataset(valid_dataset)
test_dataset = InputDataset(test_dataset)

In [9]:
# hyper-parameters
EPOCHS = 3
BATCH_SIZE = 8
LR = 1e-5

# use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# initialize the model including a classification layer with num_labels classes
# print('initializing the model')
# model = BertForTokenClassification.from_pretrained(
#     bert_version,
#     num_labels=num_labels,
#     id2label=id2label,
#     label2id=label2id
# )
# model.to(device)
# optimizer = optim.AdamW(params=model.parameters(), lr=LR)

# prepare batches of data
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)

In [10]:
# evaluate the performance of the model
def EvaluateModel(model, data_loader):
    """
    Returns:
      Y_actual_flat, Y_preds_flat : 1D numpy arrays of label ids over all
        evaluated tokens (special/subword tokens excluded). Used for token-level
        accuracy, balanced accuracy and the sklearn classification report.
      y_true_tags, y_pred_tags    : lists of lists of BIO tag strings, one
        sub-list per sentence. Used for entity-level evaluation with seqeval.
    """
    model.eval()
    Y_actual_flat, Y_preds_flat = [], []
    y_true_tags, y_pred_tags = [], []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            # move the batch tensors to the same device as the model
            batch = {k: v.to(device) for k, v in batch.items()}
            # send 'input_ids', 'attention_mask' and 'labels' to the model
            outputs = model(**batch)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)  # [batch, seq_len]

            # iterate through the examples in the batch
            for idx in range(batch['labels'].size(0)):
                true_values_all = batch['labels'][idx]
                mask = (true_values_all != -100)

                true_values = true_values_all[mask]
                pred_values = preds[idx][mask]

                # accumulate flat tensors for token-level metrics
                Y_actual_flat.append(true_values)
                Y_preds_flat.append(pred_values)

                # accumulate per-sentence BIO tag strings for seqeval
                true_tags_sent = [id2label[i] for i in true_values.tolist()]
                pred_tags_sent = [id2label[i] for i in pred_values.tolist()]
                y_true_tags.append(true_tags_sent)
                y_pred_tags.append(pred_tags_sent)

    Y_actual_flat = torch.cat(Y_actual_flat).detach().cpu().numpy()
    Y_preds_flat = torch.cat(Y_preds_flat).detach().cpu().numpy()

    return Y_actual_flat, Y_preds_flat, y_true_tags, y_pred_tags


def report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name):
    """Print both token-level and entity-level metrics."""
    print(f"\n=== {split_name} — Token-level metrics ===")
    print("Accuracy          : {:.3f}".format(accuracy_score(Y_actual, Y_preds)))
    print("Balanced accuracy : {:.3f}".format(balanced_accuracy_score(Y_actual, Y_preds)))

    print(f"\n=== {split_name} — Entity-level metrics (seqeval) ===")
    print("Precision : {:.3f}".format(seqeval_precision(y_true_tags, y_pred_tags)))
    print("Recall    : {:.3f}".format(seqeval_recall(y_true_tags, y_pred_tags)))
    print("F1        : {:.3f}".format(seqeval_f1(y_true_tags, y_pred_tags)))




In [17]:
# train the model - SKIP if model loaded
print('training the model')
for epoch in range(EPOCHS):
    model.train()
    print(f'epoch {epoch + 1}/{EPOCHS}')
    for batch in tqdm(train_loader, desc=f"Training epoch {epoch + 1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # calculate performance on validation set
    Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, valid_loader)
    report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags,
                   split_name=f"Validation (epoch {epoch + 1})")


training the model
epoch 1/3


Training epoch 1:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 1) — Token-level metrics ===
Accuracy          : 0.985
Balanced accuracy : 0.901

=== Validation (epoch 1) — Entity-level metrics (seqeval) ===
Precision : 0.927
Recall    : 0.928
F1        : 0.927
epoch 2/3


Training epoch 2:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 2) — Token-level metrics ===
Accuracy          : 0.988
Balanced accuracy : 0.938

=== Validation (epoch 2) — Entity-level metrics (seqeval) ===
Precision : 0.927
Recall    : 0.946
F1        : 0.937
epoch 3/3


Training epoch 3:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 3) — Token-level metrics ===
Accuracy          : 0.989
Balanced accuracy : 0.937

=== Validation (epoch 3) — Entity-level metrics (seqeval) ===
Precision : 0.942
Recall    : 0.950
F1        : 0.946


In [11]:
from google.colab import drive


drive.mount('/content/gdrive')
model_path = "/content/gdrive/My Drive/Colab Notebooks/saved_models"

Mounted at /content/gdrive


In [18]:
#### Save the model


# Create if path does not exist in google drive
!mkdir -p "/content/gdrive/My Drive/Colab Notebooks/saved_models"

# Save pre-trained model in path with name nlp_ass_3
model.save_pretrained(model_path)

Mounted at /content/gdrive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
# Load model from google drive
model = BertForTokenClassification.from_pretrained(
    model_path,
    # num_labels=num_labels
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# We need to align these predictions with the original words, skipping special tokens
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [18]:
print('\napplying the model to the test set')
Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, test_loader)

report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name="Test")


applying the model to the test set


Evaluating:   0%|          | 0/432 [00:00<?, ?it/s]


=== Test — Token-level metrics ===
Accuracy          : 0.980
Balanced accuracy : 0.908

=== Test — Entity-level metrics (seqeval) ===
Precision : 0.897
Recall    : 0.909
F1        : 0.903


In [19]:
# detailed token-level classification report (per-tag, including 'O')
label_ids_sorted = list(range(num_labels))
target_names = [id2label[i] for i in label_ids_sorted]
print("\n=== Test — Token-level classification report (sklearn) ===")
print(
    classification_report(
        Y_actual,
        Y_preds,
        labels=label_ids_sorted,
        target_names=target_names,
        zero_division=0
    )
)


=== Test — Token-level classification report (sklearn) ===
              precision    recall  f1-score   support

       B-LOC       0.94      0.92      0.93      1668
      B-MISC       0.83      0.83      0.83       702
       B-ORG       0.90      0.91      0.90      1661
       B-PER       0.98      0.97      0.97      1617
       I-LOC       0.81      0.89      0.85       257
      I-MISC       0.65      0.77      0.71       216
       I-ORG       0.86      0.89      0.88       835
       I-PER       0.98      0.99      0.99      1156
           O       0.99      0.99      0.99     38323

    accuracy                           0.98     46435
   macro avg       0.88      0.91      0.89     46435
weighted avg       0.98      0.98      0.98     46435



In [21]:
# detailed entity-level classification report (per entity type, no 'O')
print("=== Test — Entity-level classification report (seqeval) ===")
print(
    seqeval_report(
        y_true_tags,
        y_pred_tags,
        digits=3,
        zero_division=0
    )
)

=== Test — Entity-level classification report (seqeval) ===
              precision    recall  f1-score   support

         LOC      0.915     0.919     0.917      1668
        MISC      0.772     0.805     0.788       702
         ORG      0.862     0.886     0.874      1661
         PER      0.971     0.968     0.969      1617

   micro avg      0.897     0.909     0.903      5648
   macro avg      0.880     0.894     0.887      5648
weighted avg      0.898     0.909     0.903      5648



---

In [38]:
# Find a test sentence where the model fails to tag properly
for i in range(len(test_sentences)):
    sentence_tokens = test_sentences[i]['tokens']
    true_tags = y_true_tags[i]
    predicted_tags = y_pred_tags[i]

    if len(sentence_tokens) >= 13 and true_tags != predicted_tags:
        print(f"Sentence: {' '.join(sentence_tokens)}")
        print(f"True Tags: {true_tags}")
        print(f"Predicted Tags: {predicted_tags}")
        break

Sentence: Cuttitta announced his retirement after the 1995 World Cup , where he took issue with being dropped from the Italy side that faced England in the pool stages .
True Tags: ['B-PER', 'O', 'O', 'O', 'O', 'O', 'B-MISC', 'I-MISC', 'I-MISC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O']
Predicted Tags: ['B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'B-MISC', 'I-MISC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O']


In [39]:
matching_idx = []
missed_idx = []
for i in range(len(true_tags)):
    tt = true_tags[i]
    pt = predicted_tags[i]
    if tt == pt:
        matching_idx.append(i)
    else:
        missed_idx.append(i)

print("Matched NER tokens:")
print(list(map(lambda i: sentence_tokens[i], matching_idx)))
# print(sentence_tokens[matching_idx])

print("Missed NER tokens:")
print(list(map(lambda i: sentence_tokens[i], missed_idx)))
# print(sentence_tokens[missed_idx])

Matched NER tokens:
['Cuttitta', 'announced', 'his', 'retirement', 'after', 'the', 'Cup', ',', 'where', 'he', 'took', 'issue', 'with', 'being', 'dropped', 'from', 'the', 'Italy', 'side', 'that', 'faced', 'England', 'in', 'the', 'pool', 'stages', '.']
Missed NER tokens:
['1995', 'World']


In [40]:
classified_tokens_info = {
    "correctly_classified": [],
    "incorrectly_classified": []
}

for i in matching_idx:
    classified_tokens_info["correctly_classified"].append(
        (sentence_tokens[i], true_tags[i], predicted_tags[i])
    )

for i in missed_idx:
    classified_tokens_info["incorrectly_classified"].append(
        (sentence_tokens[i], true_tags[i], predicted_tags[i])
    )

display(classified_tokens_info)

{'correctly_classified': [('Cuttitta', 'B-PER', 'B-PER'),
  ('announced', 'O', 'O'),
  ('his', 'O', 'O'),
  ('retirement', 'O', 'O'),
  ('after', 'O', 'O'),
  ('the', 'O', 'O'),
  ('Cup', 'I-MISC', 'I-MISC'),
  (',', 'O', 'O'),
  ('where', 'O', 'O'),
  ('he', 'O', 'O'),
  ('took', 'O', 'O'),
  ('issue', 'O', 'O'),
  ('with', 'O', 'O'),
  ('being', 'O', 'O'),
  ('dropped', 'O', 'O'),
  ('from', 'O', 'O'),
  ('the', 'O', 'O'),
  ('Italy', 'B-LOC', 'B-LOC'),
  ('side', 'O', 'O'),
  ('that', 'O', 'O'),
  ('faced', 'O', 'O'),
  ('England', 'B-LOC', 'B-LOC'),
  ('in', 'O', 'O'),
  ('the', 'O', 'O'),
  ('pool', 'O', 'O'),
  ('stages', 'O', 'O'),
  ('.', 'O', 'O')],
 'incorrectly_classified': [('1995', 'B-MISC', 'O'),
  ('World', 'I-MISC', 'B-MISC')]}

In [27]:
custom_test_sentence = """
Tim Cook to become
Apple Executive Chairman

John Ternus
to become Apple CEO
"""


# Tokenize
inputs_batch_encoding = tokenizer(
    custom_test_sentence.split(), # Split into words
    is_split_into_words=True,
    return_tensors='pt',
    padding='max_length',
    truncation=True
)

# Extract word_ids
word_ids = inputs_batch_encoding.word_ids()

# Move to device
inputs = {k: v.to(device) for k, v in inputs_batch_encoding.items()}

# Get preds
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Get the predicted label IDs
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1).squeeze().tolist()

# Map the predicted IDs to labels
aligned_predictions = []
for i, pred_id in enumerate(predictions):
    if word_ids[i] is not None and (word_ids[i] != word_ids[i-1] if i > 0 else True): # Only consider first token subword
        aligned_predictions.append(id2label[pred_id])
    elif word_ids[i] is None:
        continue # Skip specials: [CLS], [SEP], [PAD]
    else:
        continue # Skip subsequent subwords I-*


print("Original Sentence:", custom_test_sentence)
print("Predicted Tags:", aligned_predictions)

Original Sentence: 
Tim Cook to become
Apple Executive Chairman

John Ternus
to become Apple CEO

Predicted Tags: ['B-PER', 'I-PER', 'O', 'O', 'B-ORG', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'B-ORG', 'O']


In [29]:
# Prettier print

custom_sentence_words = custom_test_sentence.split()
predicted_classifications = []

min_len = min(len(custom_sentence_words), len(aligned_predictions))

for i in range(min_len):
    predicted_classifications.append(
        (custom_sentence_words[i], aligned_predictions[i])
    )

print("Predicted classifications for custom sentence:")
for word, tag in predicted_classifications:
    print(f"  Word: '{word}' -> Predicted Tag: '{tag}'")

Predicted classifications for custom sentence:
  Word: 'Tim' -> Predicted Tag: 'B-PER'
  Word: 'Cook' -> Predicted Tag: 'I-PER'
  Word: 'to' -> Predicted Tag: 'O'
  Word: 'become' -> Predicted Tag: 'O'
  Word: 'Apple' -> Predicted Tag: 'B-ORG'
  Word: 'Executive' -> Predicted Tag: 'O'
  Word: 'Chairman' -> Predicted Tag: 'O'
  Word: 'John' -> Predicted Tag: 'B-PER'
  Word: 'Ternus' -> Predicted Tag: 'I-PER'
  Word: 'to' -> Predicted Tag: 'O'
  Word: 'become' -> Predicted Tag: 'O'
  Word: 'Apple' -> Predicted Tag: 'B-ORG'
  Word: 'CEO' -> Predicted Tag: 'O'


---